## Kodierung von txt-Dateien über json zu TEI/XML
### Ziele
1. Automatisierung der Kodierung von etwa 540 txt-Dateien, die Stellenanzeigen und -gesuche aus dem Sydsvenska Dagbladet des Jahres 1906 enthalten.
2. Evaluierung der Möglichkeiten (zunächst lediglich mit einem deutschen prompt für schwedische Annoncen) mit LLMs für die automatisierte Erstellung von TEI/XML-Dateien
### Bemerkungen
Der Code wurde in Anlehnung an den Code aus dem Seminar (LLM und NER) geschrieben. Da ich nicht über den anthropic API key verfüge, habe ich lediglich ollama und openai verwendet.

### importe

In [1]:
import os, json, re, textwrap, time
import requests
from openai import OpenAI
import pandas as pd
from pathlib import Path
from IPython.display import display, HTML

print("Imports erfolgreich geladen.")

Imports erfolgreich geladen.


### backend & konfiguration

In [2]:
# hier entweder "ollama" oder "openai_compat" einfügen
BACKEND = "ollama" 

# ollama
OLLAMA_BASE_URL = "http://localhost:11434/v1"
OLLAMA_MODEL    = "llama3.2"   # z.B. mistral, qwen2.5, gemma3

# openAI
OAI_BASE_URL = "http://localhost:1234/v1"
OAI_API_KEY  = "lm-studio"
OAI_MODEL    = "meta-llama-3.1-8b-instruct"

# initialisieren
if BACKEND == "ollama":
    _client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")
    _model  = OLLAMA_MODEL
elif BACKEND == "openai_compat":
    _client = OpenAI(base_url=OAI_BASE_URL, api_key=OAI_API_KEY)
    _model  = OAI_MODEL
else:
    raise ValueError(f"Unbekanntes Backend: {BACKEND!r}")

print(f"Backend : {BACKEND}")
print(f"Modell  : {_model}")

# ordner festlegen

INPUT_DIR = Path("input")
JSON_DIR = Path("json")

print("Input-Ordner:", INPUT_DIR.resolve())
print("JSON-Ordner:", JSON_DIR.resolve())


Backend : ollama
Modell  : llama3.2
Input-Ordner: C:\Users\Lenovo\digitale_editionen\input
JSON-Ordner: C:\Users\Lenovo\digitale_editionen\json


### prompt, funktionen & speichern

In [3]:
SYSTEM_PROMPT = """
Du bist ein präzises TEI-Kodierungssystem.

Du analysierst schwedische Stellenanzeigen aus dem Jahr 1906, die nicht zwingend der heutigen schwedischen Rechtschreibung und Grammatik entsprechen.
Der Text wird anhand eines vorgegebenen Kodierhandbuchs (siehe unten) semantisch
annotiert.

WICHTIGE GRUNDREGELN:

1. Der Originaltext muss vollständig erhalten bleiben.
2. Verändere keine Wörter.
3. Korrigiere keine historischen Schreibweisen.
4. Übersetze den Text nicht.
5. Füge keine Informationen hinzu.
6. Annotiere nur Informationen, die tatsächlich im Text vorkommen.
7. Verwende ausschließlich die unten angegebenen TEI-Elemente
   und Attribute.
8. Jede Annotation muss sich auf einen exakt im Originaltext
   vorkommenden Textabschnitt beziehen.
9. Die Reihenfolge des Originaltextes darf nicht verändert werden.
10. Verschachtelungen dürfen nur verwendet werden, wenn sie den
    ursprünglichen Textfluss nicht verändern.
11. Einzelne Wörter, Wortkombinationen und Teilsätze können kodiert werden. Dennoch soll die Kodierung möglichst detailliert sein.

ERLAUBTE ELEMENTE, ERLÄUTERUNGEN & ANWENDUNGSBEISPIELE:

<person>
    @sex = male | female | unspecified
    @role = employer | employee
Erläuterung: Beschreibung der Person, die eine Stelle sucht bzw. eine Stelle anbietet.
Beispiel: <person sex="female" role="employee">flicka</person> <person sex="female" role="employer">dam</person>

<person> + <occupation>
Erläuterung: Zu verwenden, wenn lediglich eine Stellenbeschreibung (und keine Personenbeschreibung) vorliegt.
Beispiel: <person sex="male" role="employee"><occupation>dräng</occupation></person> <person sex="female" role="employee"><occupation>kokerska</occupation></person>

<age>
    @type = approximate
    @value = exaktes numerisches Alter einfügen
Erläuterung: Erweiterung der Personenbeschreibung; value = genaue Altersangabe, type = ungefähre Altersangabe (wie "ung" oder "äldre")
Beispiel: <age value="25">25-årig</age> <age type="approximate">äldre</age>

<state>
    @type = marital
Erläuterung: Familienstand der suchenden/anbietenden Person
Beispiel: <state type="marital">ledig</state> <state type="marital">gift</state> <state type="marital">förlovad</state>

<trait>
    @type = character | skill | experience
Erläuterung: character bezieht sich auf Charaktereigenschaften, skill auf stellenbezogene Kenntnisse und experience auf konkrete Arbeitserfahrung (z.B. Jahresangaben)
Beispiel: <trait type="character">pålitlig och nykter</trait> <trait type="skill">van vid enkel matlagning</trait> <trait type="experience">som hade liknande plats i fyra år</trait>

<faith>
Erläuterung: Nennung einer Konfessionszugehörigkeit
Beispiel: <faith>mosaiska</faith>

<placeName>
    @type = application | employment | residence
    @ref="#..."
Erläuterung: muss keine konkrete Ortsbeschreibung sein; application bezieht sich auf Adresse, and die eine Bewerbung gesendet werden soll, employment auf den Ort, an dem die Tätigkeit ausgeübt wird, residence auf die Herkunft der Bewerber*innen; ref wird nur verwendet, wenn ein konkreter Ort benannt wird, der im Ortsregister zu finden ist (z.B. Regionen, Städte etc.)
Beispiel: <placeName type="residence" ref="#värmland">värmländska</placeName> <placeName type="employment">i mellersta Sverige</placeName> <placeName type="employment" ref="#stockholm">Stockholm</placeName>

<address> + <street>
    @type = application | employment | residence [bei <address>]
Erläuterung: vergleichbar mit <placeName>, aber bei der konkreten Adressangabe
Beispiel: <adress type="application"><street>Bondegatan 8</street> <placeName ref="#malmö">i Malmö</placeName></address>

<date>
    @type = begin | apply
    @when = konkretes Datum, sofern eindeutig bestimmbar
Erläuterung: apply bezieht sich auf die Zeit/Frist, in der man sich bewerben kann, begin auf den Anstellungsbeginn; when wird nur verwendet, wenn ein konkretes Datum genannt wird
Beispiel: <date type="begin">genast</date> <date type="apply">innan 8 dagar</date> <date type="begin" when="1906-10-24">24 oktober</date>

<measure>
    @type = salary
    @quantity = numerischer Betrag
    @unit = kronor
Erläuterung: beschreibt die Angabe von Gehalt; quantity und unit sind nur auszufüllen, wenn diese konkret genannt werden
Beispiel: <measure type="salary">någon lön önskas</measure> <measure type="salary" quantity="300" unit="kronor">årlig lön. 300 kr.</measure>

<ref> + <label>
    @type = reference | answer
Erläuterung: reference bezieht sich auf Referenzen, die gefordert/vorgebracht werden, answer (in Verbindung mit <label>) auf einen Betreff, der bei der Antwort angeführt werden soll.
Beispiel: <ref type="reference">goda ref.</ref> <ref type="reference">referenser önskas</ref> <ref type="answer">Svar till<label>"ung flicka"</label></ref> <ref type="answer">Svar till<label>"C. H."</label></ref>

<idno>
    @type = advertisement
Erläuterung: Beschreibt die Nummer/das Chiffre, das am Ende der Anzeigen stehen kann.
Beispiel: <idno type="advertisment">LB1345</idno> <idno type="advertisment">STM 124243</idno>

Gib ausschließlich ein gültiges JSON-Array zurück.

FORMAT:

[
  {
    "text": "exakt vorkommender Text",
    "start": 0,
    "end": 10,
    "element": "",
    "attributes": {}
  }
]

Die Werte von "start" und "end" sind Zeichenpositionen im
unveränderten Originaltext.

Keine Kommentare.
Kein Markdown.
Keine Erklärungen.
Nur JSON.

BEISPIELE:
a) Originaltext: "En hushållselev önskar plats å större landtgård, helst i mellersta Sverige fortast möjligt. Svar med villkor inom 8 dagar till "L.", p. r. Perstorp, Skåne. LB 3011"
[
  {
    "text": "hushållselev",
    "start": 3,
    "end": 15,
    "element": "occupation",
    "attributes": {}
  },
  {
    "text": "hushållselev",
    "start": 3,
    "end": 15,
    "element": "person",
    "attributes": {
        "sex": "unspecified",
        "role": "employee"
    }
  },
  {
    "text": "större landtgård",
    "start": 31,
    "end": 47,
    "element": "placeName",
    "attributes": {
        "type": "employment"
    }
  },
  {
    "text": "mellersta Sverige",
    "start": 57,
    "end": 74,
    "element": "placeName",
    "attributes": {
        "type": "employment"
    }
  },
  {
    "text": "fortast möjligt",
    "start": 75,
    "end": 90,
    "element": "date",
    "attributes": {
        "type": "begin"
    }
  },
  {
    "text": "med villkor",
    "start": 97,
    "end": 108,
    "element": "ref",
    "attributes": {
        "type": "reference"
    }
  },
  {
    "text": "inom 8 dagar",
    "start": 109,
    "end": 121,
    "element": "date",
    "attributes": {
        "type": "apply"
    }
  },
  {
    "text": ""L."",
    "start": 127,
    "end": 131,
    "element": "label",
    "attributes": {}
  },
  {
    "text": "p. r.",
    "start": 133,
    "end": 138,
    "element": "address",
    "attributes": {
        "type": "application"
    }
  },
  {
    "text": "Perstorp",
    "start": 139,
    "end": 147,
    "element": "placeName",
    "attributes": {
        "type": "application",
        "ref": "#perstorp"
    }
  },
  {
    "text": "Skåne",
    "start": 149,
    "end": 154,
    "element": "placeName",
    "attributes": {
        "type": "application",
        "ref": "#skane"
    }
  },
  {
    "text": "LB 3011",
    "start": 156,
    "end": 162,
    "element": "idno",
    "attributes": {
        "type": "advertisement"
    }
  }
]
b) Originaltext: "25-årig kammarjungfru, med vackra betyg från förnäma hus, söker plats. A. Bulling. Malmö Rt. 349. (STM. 123842)"
[
  {
    "text": "25-årig",
    "start": 0,
    "end": 7,
    "element": "age",
    "attributes": {
        "value": "25"
    }
  },
  {
    "text": "kammarjungfru",
    "start": 8,
    "end": 21,
    "element": "person",
    "attributes": {
        "sex": "female",
        "role": "employee"
    }
  },
  {
    "text": "kammarjungfru",
    "start": 8,
    "end": 21,
    "element": "occupation",
    "attributes": {}
  },
  {
    "text": "vackra betyg från förnäma hus",
    "start": 27,
    "end": 56,
    "element": "ref",
    "attributes": {
        "type": "reference"
    }
  },
  {
    "text": "A. Bulling",
    "start": 71,
    "end": 81,
    "element": "person",
    "attributes": {
        "role": "employer"
    }
  },
  {
    "text": "Malmö",
    "start": 83,
    "end": 88,
    "element": "placeName",
    "attributes": {
        "type": "employment",
        "ref": "#malmö"
    }
  },
  {
    "text": "Rt. 349",
    "start": 89,
    "end": 96,
    "element": "idno",
    "attributes": {
        "type": "advertisement"
    }
  },
  {
    "text": "STM. 123842",
    "start": 99,
    "end": 110,
    "element": "idno",
    "attributes": {
        "type": "advertisement"
    }
  }
]
"""

# funktionen zur erstellung des jsons
def parse_json(raw):
    raw = raw.strip()

    # falls es doch markdown gibt
    raw = re.sub(r"^```(?:json)?\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)

    return json.loads(raw)

def annotate_text(text, filename):
    
    user_prompt = f"""
Dateiname:
{filename}

Originaltext:
{text}
"""

    response = _client.chat.completions.create(
        model= _model,
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]
    )

    raw = response.choices[0].message.content

    return parse_json(raw)


# speichern der dateien
def process_file(txt_path):
    
    # originaltext lesen
    text = txt_path.read_text(encoding="utf-8")
    
    # dateiname ohne .txt
    file_id = txt_path.stem
    
    print(f"Verarbeite: {file_id}")
    
    # LLM aufrufen
    annotations = annotate_text(text, txt_path.name)
    
    # JSON-Datei erzeugen
    json_path = JSON_DIR / f"{file_id}.json"
    
    result = {
        "id": file_id,
        "source_file": txt_path.name,
        "text": text,
        "annotations": annotations
    }
    
    with json_path.open("w", encoding="utf-8") as f:
        json.dump(
            result,
            f,
            ensure_ascii=False,
            indent=2
        )
    
    print(f"Gespeichert: {json_path}")
    
    return result

### test-datei

In [ ]:
# Alle TXT-Dateien aus dem Input-Ordner einlesen
txt_files = sorted(INPUT_DIR.glob("*.txt"))

print(f"{len(txt_files)} Dateien gefunden.")

for i, txt_file in enumerate(txt_files, start=1):
    print(f"\n===== Datei {i}/{len(txt_files)} =====")
    process_file(txt_file)

7 Dateien gefunden.

===== Datei 1/7 =====
Verarbeite: 1906_01_24_SD_PS_001


In [ ]:
c) Originaltext: "Hushållerska riktigt skicklig och kompetent att sköta landthushåll, försedd med bästa rekommendationer erhåller plats å Villa Åsum d. 24 April. Svar torde sändas efter den 6 Febr. till friherrinnan E. Ramel adress Villa Åsum Sjöbo."
[
  {
    "text": "Hushållerska",
    "start": 0,
    "end": 12,
    "element": "person",
    "attributes": {
        "sex": "female",
        "role": "employee"
    }
  }
]
[
  {
    "text": "Hushållerska",
    "start": 0,
    "end": 12,
    "element": "occupation",
    "attributes": {}
  }
]
[
  {
    "text": "riktig skicklig",
    "start": 13,
    "end": 29,
    "element": "trait",
    "attributes": {
        "type": "character"
    }
  }
]
[
  {
    "text": "kompetent att sköta landthushåll",
    "start": 34,
    "end": 66,
    "element": "trait",
    "attributes": {
        "type": "skill"
    }
  }
]
[
  {
    "text": "bästa rekommendationer",
    "start": 80,
    "end": 102,
    "element": "ref",
    "attributes": {
        "type": "reference"
    }
  }
]
[
  {
    "text": "Villa Åsum",
    "start": 120,
    "end": 130,
    "element": "placeName",
    "attributes": {
        "type": "employment"
    }
  }
]
[
  {
    "text": "24 April",
    "start": 134,
    "end": 142,
    "element": "date",
    "attributes": {
        "when": "1906-04-24",
        "type": "begin"
    }
  }
]
[
  {
    "text": "efter den 6 Febr.",
    "start": 162,
    "end": 178,
    "element": "date",
    "attributes": {
        "type": "apply",
        "notBefore": "1906-02-06"
    }
  }
]
[
  {
    "text": "friherrinnan E. Ramel",
    "start": 185,
    "end": 206,
    "element": "person",
    "attributes": {
        "role": "employer",
        "sex": "female"
    }
  }
]
[
  {
    "text": "Villa Åsum",
    "start": 214,
    "end": 224,
    "element": "placeName",
    "attributes": {
        "type": "employment"
    }
  }
]
[
  {
    "text": "Sjöbo",
    "start": 225,
    "end": 230,
    "element": "placeName",
    "attributes": {
        "type": "employment",
        "ref": "#sjöbo"
    }
  }
]
d) Originaltext: "Hushållerskor, kokerskor (lön 200–300), kammarjungfrur (225), vana städerskor (150–200), tvätterskor (180–200) barn- & ensamjungfrur (100–200), m. fl. erhålla praktiga platser den 28 april i Malmö, andra städer och orter, slott och herregårdar. Insänd bet. snarast. Svar mot 20 öre. Rydowsa agentur, 6 Davidshallsgat., Malmö. STM 125922."
[
  {
    "text": "Hushållerskor",
    "start": 0,
    "end": 13,
    "element": "person",
    "attributes": {
        "sex": "female",
        "role": "employee"
    }
  }
]
[
  {
    "text": "Hushållerskor",
    "start": 0,
    "end": 13,
    "element": "occupation",
    "attributes": {}
  }
]
[
  {
    "text": "kokerskor",
    "start": 15,
    "end": 24,
    "element": "person",
    "attributes": {
        "sex": "female",
        "role": "employee"
    }
  }
]
[
  {
    "text": "kokerskor",
    "start": 15,
    "end": 24,
    "element": "occupation",
    "attributes": {}
  }
]
[
  {
    "text": "200–300",
    "start": 30,
    "end": 37,
    "element": "measure",
    "attributes": {
        "type": "salary",
        "quantity": "200–300",
        "unit": "kronor"
    }
  }
]
[
  {
    "text": "kammarjungfrur",
    "start": 40,
    "end": 54,
    "element": "person",
    "attributes": {
        "sex": "female",
        "role": "employee"
    }
  }
]
[
  {
    "text": "kammarjungfrur",
    "start": 40,
    "end": 54,
    "element": "occupation",
    "attributes": {}
  }
]
[
  {
    "text": "225",
    "start": 56,
    "end": 59,
    "element": "measure",
    "attributes": {
        "type": "salary",
        "quantity": "225",
        "unit": "kronor"
    }
  }
]
[
  {
    "text": "tvätterskor",
    "start": 89,
    "end": 100,
    "element": "person",
    "attributes": {
        "sex": "female",
        "role": "employee"
    }
  }
]
[
  {
    "text": "tvätterskor",
    "start": 89,
    "end": 100,
    "element":  "occupation",
    "attributes": {}
  }
]
[
  {
    "text": "150–200",
    "start": 79,
    "end": 86,
    "element": "measure",
    "attributes": {
        "type": "salary",
        "quantity": "150–200",
        "unit": "kronor"
    }
  }
]
[
  {
    "text": "städerskor",
    "start": 67,
    "end": 77,
    "element": "person",
    "attributes": {
        "sex": "female",
        "role": "employee"
    }
  }
]
[
  {
    "text": "städerskor",
    "start": 67,
    "end": 77,
    "element": "occupation",
    "attributes": {}
  }
]
[
  {
    "text": "180–200",
    "start": 102,
    "end": 109,
    "element": "measure",
    "attributes": {
        "type": "salary",
        "quantity": "180–200",
        "unit": "kronor"
    }
  }
]
[
  {
    "text": "barn- & ensamjungfrur",
    "start": 111,
    "end": 132,
    "element": "person",
    "attributes": {
        "sex": "female",
        "role": "employee"
    }
  }
]
[
  {
    "text": "barn- & ensamjungfrur",
    "start": 111,
    "end": 132,
    "element": "occupation",
    "attributes": {}
  }
]
[
  {
    "text": "100–200",
    "start": 134,
    "end": 141,
    "element": "measure",
    "attributes": {
        "type": "salary",
        "quantity": "100–200",
        "unit": "kronor"
    }
  }
]
[
  {
    "text": "28 april",
    "start": 180,
    "end": 188,
    "element": "date",
    "attributes": {
        "type": "begin",
        "when": "1906-04-28"
    }
  }
]
[
  {
    "text": "Malmö",
    "start": 191,
    "end": 196,
    "element": "placeName",
    "attributes": {
        "type": "employment",
        "ref": "#malmö"
    }
  }
]
[
  {
    "text": "andra städer och orter, slott och herregårdar",
    "start": 198,
    "end": 243,
    "element": "placeName",
    "attributes": {
        "type": "employment"
    }
  }
]
[
  {
    "text": "snarast",
    "start": 257,
    "end": 264,
    "element": "date",
    "attributes": {
        "type": "apply"
    }
  }
]
[
  {
    "text": "mot 20 öre",
    "start": 272,
    "end": 284,
    "element": "measure",
    "attributes": {
        "type": "salary",
        "quantity": "20",
        "unit": "öre"
    }
  }
]
[
  {
    "text": "Rydowsa Agentur",
    "start": 286,
    "end": 301,
    "element": "placeName",
    "attributes": {
        "type": "application"
    }
  }
]
[
  {
    "text": "6 Davidshallgat.",
    "start": 303,
    "end": 319,
    "element": "placeName", "address", "street"
    "attributes": {
        "type": "application"
    }
  }
]
[
  {
    "text": "Malmö",
    "start": 322,
    "end": 327,
    "element": "placeName",
    "attributes": {
        "type": "application",
        "ref": "#malmö"
    }
  }
]
[
  {
    "text": "STM 125922",
    "start": 329,
    "end": 339,
    "element": "idno",
    "attributes": {
        "type": "advertisement"
    }
  }
]
"""

e) Originaltext: "Ung flicka, önskar komma i god och vänlig familj att vara frun behjälplig med lättare inomhus förekommande göromål, helst der tillfälle gifves något lära hushåll. Tacksam för svar till 'Postfack 32', Hvellinge. (GAM. 141912)"
[
  {
    "text": "ung",
    "start": 0,
    "end": 3,
    "element": "age",
    "attributes": {
        "type": "approximate"
    }
  }
]
[
  {
    "text": "flicka",
    "start": 4,
    "end": 10,
    "element": "person",
    "attributes": {
        "sex": "female",
        "type": "employee"
    }
  }
]
[
  {
    "text": "god och vänlig familj",
    "start": 27,
    "end": 48,
    "element": "placeName",
    "attributes": {
        "type": "employment"
    }
  }
]
[
  {
    "text": "frun behjälplig med lättare inomhus förekommande göromål",
    "start": 58,
    "end": 114,
    "element": "trait",
    "attributes": {
        "type": "skill"
    }
  }
]
[
  {
    "text": "lära hushåll",
    "start": 149,
    "end": 161,
    "element": "trait",
    "attributes": {
        "type": "experience"
    }
  }
]
[
  {
    "text": "'Postfack 32'",
    "start": 185,
    "end": 198,
    "element": "label",
    "attributes": {
    }
  }
]
[
  {
    "text": "Hvellinge",
    "start": 200,
    "end": 209,
    "element": "placeName",
    "attributes": {
        "type": "application"
        "ref": "#hvellinge"
    }
  }
]
[
  {
    "text": "GAM. 141912",
    "start": 212,
    "end": 223,
    "element": "idno",
    "attributes": {
        "type": "advertisement"
    }
  }
]
f) Originaltext: "För en ordentlig och i matlagning kunnig kokerska och en städerska finnes god plats i liten familj den 24 Okt. Svar till Hessleholm p. r."
[
  {
    "text": "ordentlig",
    "start": 7,
    "end": 16,
    "element": "trait",
    "attributes": {
        "type": "character"
    }
  }
]
[
  {
    "text": "i matlagning kunnig",
    "start": 21,
    "end": 40,
    "element": "trait",
    "attributes": {
        "type": "skill"
    }
  }
]
[
  {
    "text": "kokerska",
    "start": 41,
    "end": 49,
    "element": "person", "occupation",
    "attributes": {
        "sex": "female",
        "type": "employee"
    }
  }
]
[
  {
    "text": "städerska",
    "start": 57,
    "end": 66,
    "element": "person", "occupation",
    "attributes": {
        "sex": "female",
        "type": "employee"
    }
  }
]
[
  {
    "text": "liten familj",
    "start": 96,
    "end": 108,
    "element": "placeName",
    "attributes": {
        "type": "employment"
    }
  }
]
[
  {
    "text": "24 okt.",
    "start": 113,
    "end": 119,
    "element": "date",
    "attributes": {
        "when": "1906-10-24",
        "type": "begin"
    }
  }
]
[
  {
    "text": "Hessleholm",
    "start": 131,
    "end": 141,
    "element": "placeName",
    "attributes": {
        "ref": "#hessleholm",
        "type": "application"
    }
  }
]
[
  {
    "text": "p. r.",
    "start": 142,
    "end": 146,
    "element": "address",
    "attributes": {
        "type": "application"
    }
  }
]
g) Originaltext: "Husföreståndarinna, fullt kunnig i matlagning och allt som förekommer i ett väl ordnadt hem, söker plats i Malmö. Svar till 'Jämnt lynne 35 år', Gumælii Annonsbyrå, Malmö f. v. b. GAM 146149"
[
  {
    "text": "husföreståndarinna",
    "start": 0,
    "end": 18,
    "element": "person", "occupation",
    "attributes": {
        "sex": "female",
        "type": "employee"
    }
  }
]
[
  {
    "text": "fullt kunnig i matlagning och allt som förekommer i ett väl ordnadt hem",
    "start": 20,
    "end": 91,
    "element": "trait",
    "attributes": {
        "type": "skill"
    }
  }
]
[
  {
    "text": "Malmö",
    "start": 107,
    "end": 112,
    "element": "placeName",
    "attributes": {
        "ref": "#malmö",
        "type": "employment"
    }
  }
]
[
  {
    "text": "'Jämnt lynne 35 år'",
    "start": 0,
    "end": 18,
    "element": "label",
    "attributes": {}
  }
]
[
  {
    "text": "Gumælii Annonsbyrå",
    "start": 145,
    "end": 163,
    "element": "placeName",
    "attributes": {
        "type": "application"
    }
  }
]
[
  {
    "text": "Malmö",
    "start": 165,
    "end": 170,
    "element": "label",
    "attributes": {
        "type": "application",
        "ref": "#malmö"
    }
  }
]
[
  {
    "text": "GAM 146149",
    "start": 180,
    "end": 190,
    "element": "idno",
    "attributes": {
        "type": "advertisement"
    }
  }
]
h) Originaltext: "Utmärkta platser. För skickliga landthushållerskor, hemma i finare matlagning, konservering m. m. Vid. gen Th. Tjäders Byrå, Brunkebergstorg 12, Stockholm. STA 174359"
i) Originaltext: "Kokerska till Köpenhamn. En i finare matlagning fullt kompetent kokerska erhåller förmånlig plats den 1 eller 24 Oktober i svensk familj i Köpenhamn, 2 personer. Ingen utan de bästa rekomendationer torde anmäla sig. Adress Fru G. Barclay, Strandvejen 4, Köpenhamn."
"""